# `ptof_obs_mal_output`

## What this notebook does
Detects two distinct ways an agent output can be malformed:
1. **Blank output** (CRITICAL) — the output record exists but content is empty, null, or trivially
   hollow. Invisible to every other detector because the record looks like it arrived normally.
2. **Response schema drift** (CRITICAL) — an expected JSON field in the output silently stopped
   appearing, compared against the nightly baseline.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `03_malformed_output` — runs after `01_bronze_projections`,
  before `06_alert`.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`), and
  `response_field_baseline` (built nightly by `ptof_obs_nightly_baseline`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `blank_output_findings` (CRITICAL) and
  `response_schema_drift` (CRITICAL, filtered to `drift_type = 'field_missing'`).

## Tables/views touched
- **Reads:** `v_llm_bronze`, `capability_registry`, `response_field_baseline`
- **Writes:** `blank_output_findings`, `response_schema_drift`

## Dropped detectors (prod migration 2026-09-10)
- `transport_violation_signatures` — single transport (`cortex`), zero violations ever recorded

In [ ]:
%python
# blank_output_findings — aggregated over 6h window for MERGE dedup. See markdown cell above
# for full history/basis. Refactored 2026-09-22 (backtest support) into a def ..._sql(as_of)
# builder so ptof_obs_backtest_thresholds.ipynb can evaluate this detector at any historical
# point -- called below with the default as_of="current_timestamp()" so production behavior is
# unchanged.
def blank_output_findings_sql(as_of="current_timestamp()"):
    return f"""
    CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.blank_output_findings AS
    WITH blank_output_incidents AS (
      SELECT
          date_trunc('HOUR', b.called_at) AS hour,
          b.capability, b.model_config,
          count_if(b.is_blank_output) AS blank_output_count,
          count(*)                    AS total_calls,
          count_if(b.is_blank_output) * 1.0 / count(*) AS blank_output_rate
      FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
      JOIN mq_gmdf_dev.oil_obs.capability_registry r
        ON r.capability = b.capability AND r.active = true
      WHERE b.called_at >= {as_of} - INTERVAL 7 DAYS
      GROUP BY 1, 2, 3
      HAVING count_if(b.is_blank_output) > 0
    )
    SELECT
        capability, model_config,
        sum(blank_output_count) AS blank_count_window,
        sum(total_calls) AS total_calls_window,
        round(sum(blank_output_count) * 1.0 / nullif(sum(total_calls), 0), 4) AS blank_rate_window,
        max(hour) AS latest_hour,
        sha2(concat_ws('|',
            coalesce(capability, '<null>'),
            coalesce(model_config, '<null>')
        ), 256) AS finding_signature,
        {as_of} AS detected_at
    FROM blank_output_incidents
    WHERE hour >= date_trunc('HOUR', {as_of} - INTERVAL 6 HOURS)
    GROUP BY capability, model_config
    HAVING sum(blank_output_count) >= 1
    """

spark.sql(blank_output_findings_sql())

In [ ]:
%python
# response_schema_drift — a field that used to reliably appear in a capability's response
# silently disappearing. See markdown cell above for full history/basis. Refactored into a
# def ..._sql(as_of) builder, same pattern as blank_output_findings above.
def response_schema_drift_sql(as_of="current_timestamp()"):
    return f"""
    CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_schema_drift AS
    WITH current_keys AS (
      SELECT b.capability, k.key AS field_name, count(*) AS current_present,
             collect_set(b.model_config) AS model_configs_seen
      FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
      JOIN mq_gmdf_dev.oil_obs.capability_registry r
        ON r.capability = b.capability AND r.active = true
      LATERAL VIEW explode(from_json(cast(b.response_parsed AS STRING), 'map<string,string>')) k AS key, value
      WHERE b.is_blank_output = false
        AND b.called_at >= {as_of} - INTERVAL 24 HOURS
      GROUP BY b.capability, k.key
    ),
    current_rows AS (
      SELECT b.capability, count(*) AS n_rows
      FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
      JOIN mq_gmdf_dev.oil_obs.capability_registry r
        ON r.capability = b.capability AND r.active = true
      WHERE b.is_blank_output = false
        AND b.called_at >= {as_of} - INTERVAL 24 HOURS
      GROUP BY b.capability
    ),
    comparison AS (
      SELECT
          bk.capability,
          bk.field_name,
          bk.baseline_present,
          bk.baseline_total AS baseline_rows,
          bk.baseline_presence_rate,
          coalesce(ck.current_present, 0) AS current_present,
          ck.model_configs_seen,
          cr.n_rows AS current_rows,
          'field_missing' AS drift_type,
          true AS schema_changed
      FROM mq_gmdf_dev.oil_obs.response_field_baseline bk
      LEFT JOIN current_keys ck
        ON ck.capability = bk.capability AND ck.field_name = bk.field_name
      LEFT JOIN current_rows cr ON cr.capability = bk.capability
    )
    SELECT
        capability,
        field_name,
        baseline_present,
        baseline_rows,
        baseline_presence_rate,
        current_present,
        model_configs_seen,
        current_rows,
        drift_type,
        schema_changed,
        sha2(concat_ws('|',
            coalesce(capability, '<null>'),
            coalesce(field_name, '<null>')
        ), 256) AS finding_signature,
        {as_of} AS detected_at
    FROM comparison
    WHERE current_rows >= 10
      AND current_present = 0
      AND baseline_presence_rate >= 0.2
    """

spark.sql(response_schema_drift_sql())